# ES-MoE quick start

**[esmoe](https://github.com/Lfan-ke/ES-MoE)** adds an expert-sparse Mixture-of-Experts block to
Ultralytics YOLO. It installs *beside* the official `ultralytics` package — no fork, no patched
library — and it wires the router's load-balancing loss into the loss the optimiser actually sees.

This notebook takes about five minutes on a free Colab GPU (`Runtime -> Change runtime type -> T4`),
and works on CPU too, just slower.

The outputs stored below are from one real execution of this file on a CPU machine, kept so you can
see what to expect before running anything. Your numbers will differ: eight images and three epochs
are not a measurement.

| step | what you will see |
|:--:|:--:|
| 1 | the block inside a real YOLO11 model |
| 2 | an `esmoe_aux` column appearing in the training log |
| 3 | a same-budget comparison, and why you should not trust it |
| 4 | your own expert and balancing objective, in ten lines |
| 5 | several blocks at once, and the command line |

## 0. Setup

One package, plus pandas for the log table in step 2; Colab already has pandas.

In [1]:
import importlib.util
import subprocess
import sys

missing = [name for name in ("esmoe", "pandas") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [2]:
import torch
import ultralytics

import esmoe

print(f"esmoe        {esmoe.__version__}")
print(f"ultralytics  {ultralytics.__version__}")
print(f"torch        {torch.__version__}")
print(f"cuda         {torch.cuda.is_available()}")

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\esmoe-run\cfg\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


esmoe        1.0.0
ultralytics  8.4.132
torch        2.13.0+cpu
cuda         False


## 1. Put the block in a model

`equip` does four things in one call:

1. **registers** `ESMoE` so a `model.yaml` may name it,
2. **grafts** it onto the backbone and renumbers every head reference the insertion would break,
3. **builds** the model,
4. **attaches** the auxiliary loss to training.

The printed block tells you what you got: the inferred channel count, four experts, top-2 routing,
and one kernel size per expert — the experts differ in receptive field, not only in weights.

In [3]:
model = esmoe.equip("yolo11n.yaml", weight=0.01)

block = next(esmoe.blocks(model.model))
print(block)

ESMoE(
  channels=256, num_experts=4, top_k=2, kernels=[3, 5, 7, 9], balance=switch, out_norm=False, dense_training=False, sparse_inference=True, dynamic_threshold=0.0
  (experts): ModuleList(
    (0): DWExpert(
      (dw): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=256, bias=False)
      (pw): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): DWExpert(
      (dw): Conv2d(256, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), groups=256, bias=False)
      (pw): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): DWExpert(
      (dw): Conv2d(256, 256, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=256, bias=False)


## 2. Train, and watch the auxiliary loss

`coco8` is Ultralytics' eight-image toy dataset. It downloads in seconds and exists for plumbing
checks, not for accuracy.

The column to watch is **`train/esmoe_aux`**. Its presence is the whole point: the router's
load-balancing term is inside the loss that gets back-propagated, not merely computed and logged.

In [4]:
result = model.train(
    data="coco8.yaml",
    epochs=3,
    imgsz=320,
    batch=4,
    workers=2,
    plots=False,
    name="quickstart-esmoe",
    exist_ok=True,
    verbose=False,
)

print(f"trained. mAP50 = {result.box.map50:.4f}, run directory = {model.trainer.save_dir}")

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'


Ultralytics 8.4.132  Python-3.12.12 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\esmoe-run\tmp\esmoe-hapi3avv\yolo11n-esmoe.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=quickstart-esmoe, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

WARNING Dataset 'coco8.yaml' images not found, missing path 'C:\esmoe-run\work\datasets\coco8\images\val'


Unzipping C:\esmoe-run\work\datasets\coco8.zip to C:\esmoe-run\work\datasets\coco8...: 100% ━━━━━━━━━━━━ 25/25 1.1Kfiles/s 0.0s

...

YOLO11n-esmoe summary (fused): 123 layers, 2,930,780 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 10.9it/s 0.1s

                   all          4         17          0          0          0          0


Speed: 0.2ms preprocess, 20.4ms inference, 0.0ms loss, 0.1ms postprocess per image


trained. mAP50 = 0.0000, run directory = C:\esmoe-run\work\runs\detect\quickstart-esmoe


In [5]:
import pandas as pd

history = pd.read_csv(model.trainer.save_dir / "results.csv")
history[[column for column in history.columns if "loss" in column or "esmoe" in column]]

,train/box_loss,train/cls_loss,train/dfl_loss,train/esmoe_aux,val/box_loss,val/cls_loss,val/dfl_loss,val/esmoe_aux
0,3.33309,5.69213,4.29576,0.02097,3.22478,5.83479,4.15889,0.02097
1,3.58616,5.66931,4.26418,0.02100,3.22478,5.83479,4.15889,0.02097
2,3.27672,5.69717,4.25392,0.02081,3.22478,5.83479,4.15889,0.02097


## 3. A same-budget comparison

Identical data, schedule, batch and seed; only the block differs. That is the only kind of
comparison worth making.

On eight images the gap is noise, and the cell below says so. The real
evidence, paired seeds on VisDrone under the repository protocol, is on the
[judgment lines](https://lfan-ke.github.io/ES-MoE/en/JUDGMENT/) and [results](https://lfan-ke.github.io/ES-MoE/en/results/) pages.

In [6]:
from ultralytics import YOLO

baseline = YOLO("yolo11n.yaml")
_ = baseline.train(
    data="coco8.yaml",
    epochs=3,
    imgsz=320,
    batch=4,
    workers=2,
    plots=False,
    name="quickstart-baseline",
    exist_ok=True,
    verbose=False,
)

for label, trained in (("baseline", baseline), ("esmoe", model)):
    print(f"{label:<9} mAP50 = {trained.trainer.metrics['metrics/mAP50(B)']:.4f}")

print("\nEight images, three epochs: a plumbing check, not evidence.")

New https://pypi.org/project/ultralytics/8.4.152 available  Update with 'pip install -U ultralytics'


Ultralytics 8.4.132  Python-3.12.12 torch-2.13.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)


engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=quickstart-baseline, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, pl


                   from  n    params  module                                       arguments                     


  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 


...

YOLO11n summary (fused): 101 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 9.9it/s 0.1s

                   all          4         17          0          0          0          0


Speed: 0.2ms preprocess, 22.2ms inference, 0.0ms loss, 0.1ms postprocess per image


baseline  mAP50 = 0.0000
esmoe     mAP50 = 0.0000

Eight images, three epochs: a plumbing check, not evidence.


## 4. Bring your own expert

Both extension points are plain callables:

- `expert(c1, c2, k) -> Module` builds one branch,
- `balance(probs, gate) -> scalar` scores how evenly the router spreads its traffic.

Swap either and the rest keeps working. Here the experts become plain grouped convolutions and the
balancing term becomes routing entropy.

In [7]:
from torch import nn


class ThinExpert(nn.Sequential):
    def __init__(self, c1, c2, k):
        super().__init__(nn.Conv2d(c1, c2, k, 1, k // 2, groups=c1), nn.SiLU())


def entropy_balance(probs, gate):
    return -(probs * probs.clamp_min(1e-9).log()).sum(dim=1).mean()


custom = esmoe.ESMoE(num_experts=3, top_k=2, expert=ThinExpert, balance=entropy_balance)
custom(torch.randn(2, 32, 16, 16))

print("experts   ", [type(expert).__name__ for expert in custom.experts])
print("kernels   ", custom.expert_kernel_sizes)
print("aux loss  ", f"{esmoe.collect_aux_loss(custom).item():.4f}")

experts    ['ThinExpert', 'ThinExpert', 'ThinExpert']
kernels    [3, 5, 7]
aux loss   1.0856


## 5. Several blocks, and the command line

`at` accepts the end of the backbone (the default), one layer index, or several. Every reference
after an insertion point is renumbered for you.

In [8]:
config = esmoe.graft("yolo11n.yaml", at=[4, 6], num_experts=4, top_k=2)

positions = [i for i, layer in enumerate(config["backbone"]) if layer[2] == "ESMoE"]
print("ESMoE at backbone layers", positions)

ESMoE at backbone layers [5, 8]


In [9]:
!esmoe graft yolo11n.yaml -o yolo11n-esmoe.yaml -e 4 -k 2 --at backbone_end
!esmoe info

'esmoe' is not recognized as an internal or external command,
operable program or batch file.


'esmoe' is not recognized as an internal or external command,
operable program or batch file.


## 6. Match upstream, and why it goes in the config

The block defaults to what the matrix runs in `results/` were measured with. Upstream's `ES_MOE`
differs in four places that affect training and one that affects inference, and every one of
them is a keyword on `equip`:

| setting | here | upstream |
|:--:|:--:|:--:|
| `at="backbone_stages"` | one block at the backbone end | one after each stage, four in all |
| `balance="gshard"` | `switch_balance`, reads the full softmax | reads the gate, after the top-k renormalisation |
| `out_norm` | off | `BatchNorm + SiLU` after the weighted sum |
| `dense_training` | off | every expert runs while training |
| `dynamic_threshold` | `0.0` | `0.4`, pruning low-share experts at inference |

The same-configuration runs against YOLO-Master also set `recipe="upstream"` with `weight=1.0`:
the auxiliary term is normalised by its running magnitude and capped, routers get half the
learning rate, and experts stay frozen for three epochs.

They go through `equip` rather than onto the blocks afterwards, because the trainer rebuilds
the model from its config: anything set on the instance goes with the instance it discards.
`block.spec()` reports what a block is holding, and `scripts/blockspec.py` reads it back from
any checkpoint — which is how a record here can state what trained rather than what was asked for.


In [10]:
aligned = esmoe.equip(
    "yolo11n.yaml",
    at="backbone_stages",
    balance="gshard",
    out_norm=True,
    dense_training=True,
    dynamic_threshold=0.4,
    recipe="upstream",
    weight=1.0,
)

blocks = list(esmoe.blocks(aligned.model))
print("blocks   ", len(blocks))
print("settings ", blocks[0].spec())

blocks    4
settings  {'balance': 'gshard', 'out_norm': True, 'dense_training': True, 'sparse_inference': True, 'dynamic_threshold': 0.4}


## Where to go next

- **[Documentation](https://lfan-ke.github.io/ES-MoE/en/)** — tutorial and API; the Chinese edition is at the site root.
- **[Selection evidence](https://github.com/Lfan-ke/ES-MoE/blob/main/docs/SELECTION.en.md)** — why four
  experts, top-2 and an auxiliary weight of 0.01 are the shipped defaults.
- **[Limitations](https://lfan-ke.github.io/ES-MoE/en/limitations/)** — read this before
  quoting any number from the project. The gain depends on the backbone and turns negative on YOLO12n and YOLO26n.
- `scripts/sweep.sh` in the repository reproduces the multi-seed comparison on a real dataset.